# 🔬 TSM — Text Similarity Maker

Generate embeddings of your documents' titles and abstracts, then build a science map.

**Instructions:**
1. Run **Step 1** to install dependencies (~2 min, only needed once per session)
2. Run **Step 2** to launch the app
3. The app will appear below — use it just like the web version

> Built by [Juan Pablo Bascur](https://jpbascur.com)

In [ ]:
#@title Step 1 — Install dependencies (run once) { display-mode: "form" }
%%capture
!pip install streamlit transformers adapters torch umap-learn numpy pandas psutil
!git clone https://github.com/jpbascur/text-similarity-maker.git /content/tsm 2>/dev/null || git -C /content/tsm pull
print('✅ Done. Run Step 2 to launch the app.')

In [ ]:
#@title Step 2 — Launch app { display-mode: "form" }
import subprocess, time, re, urllib.request, os
from IPython.display import display, IFrame

# Install cloudflared
subprocess.run(
    ['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
     '-O', '/usr/local/bin/cloudflared'], check=True
)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

# Start Streamlit, logging to file for debugging
log_file = open('/tmp/streamlit.log', 'w')
subprocess.Popen(
    ['streamlit', 'run', '/content/tsm/streamlit_app.py',
     '--server.port=8501', '--server.headless=true'],
    env={**os.environ, 'TSM_COLAB': '1'},
    stdout=log_file, stderr=log_file
)

# Wait until Streamlit is actually ready
print('Starting Streamlit…')
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:8501', timeout=1)
        print('Streamlit is up.')
        break
    except:
        time.sleep(1)
else:
    print('⚠️ Streamlit failed to start. Run the Debug cell below.')

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

url = None
for line in tunnel.stdout:
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print(f'✅ App running at: {url}')
display(IFrame(url, width='100%', height=800))

In [ ]:
#@title Toy test — can Streamlit open in Colab at all?
import subprocess, time, re, urllib.request, os
from IPython.display import display, IFrame

# Kill any existing Streamlit
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(1)

# Write a minimal Streamlit app
with open('/tmp/toy_app.py', 'w') as f:
    f.write("import streamlit as st\nst.title('It works!')\nst.write('Streamlit is running in Colab.')\n")

# Install cloudflared to /tmp (avoids permission issues)
cf_path = '/tmp/cloudflared'
result = subprocess.run(
    ['curl', '-sL',
     'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
     '-o', cf_path])
subprocess.run(['chmod', '+x', cf_path])
print('cloudflared downloaded:', os.path.exists(cf_path))

# Start toy app on fixed port
subprocess.Popen(
    ['streamlit', 'run', '/tmp/toy_app.py', '--server.port=8501', '--server.headless=true'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Wait until ready
for _ in range(20):
    try:
        urllib.request.urlopen('http://localhost:8501', timeout=1)
        print('Streamlit is up.')
        break
    except:
        time.sleep(1)

# Tunnel
tunnel = subprocess.Popen(
    [cf_path, 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
url = None
for line in tunnel.stdout:
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print(f'URL: {url}')
display(IFrame(url, width='100%', height=400))

In [ ]:
#@title Debug — check if Streamlit started correctly { display-mode: "form" }
import subprocess
result = subprocess.run(['curl', '-s', '-o', '/dev/null', '-w', '%{http_code}', 'http://localhost:8501'], capture_output=True, text=True)
print('Streamlit HTTP status:', result.stdout)

log = subprocess.run(['cat', '/tmp/streamlit.log'], capture_output=True, text=True)
print(log.stdout[-3000:] if log.stdout else 'No log found')